In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_31428/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Pascal Siakam,Over,22.5,-137,2025-11-22,2025-11-21T23:32:12Z
1,Underdog,player_points,Pascal Siakam,Under,22.5,-137,2025-11-22,2025-11-21T23:32:12Z
2,Underdog,player_points,Darius Garland,Over,15.5,-137,2025-11-22,2025-11-21T23:32:12Z
3,Underdog,player_points,Darius Garland,Under,15.5,-137,2025-11-22,2025-11-21T23:32:12Z
4,Underdog,player_points,Donovan Mitchell,Over,28.5,-137,2025-11-22,2025-11-21T23:32:12Z


### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

singleBets = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)



singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION', 'SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets...
Pre-computing predictions for 139 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
189,Jerami Grant,DraftKings,22.5,17.10,Under,-111,1,6.09,0.676,Med
1140,Tre Jones,BetMGM,9.5,14.89,Over,-110,1,5.71,0.628,Med
854,Alperen Sengun,BetRivers,24.5,28.14,Over,112,0,5.09,0.455,High
1060,Bennedict Mathurin,BetMGM,21.5,25.16,Over,110,0,5.07,0.461,High
523,Evan Mobley,BetRivers,18.5,14.84,Under,108,0,5.06,0.469,High


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 99 players...
Processing 91 players...
Generated 3890 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
2596,Tre Jones,Jerami Grant,9.5,22.5,14.89,17.10,0.823,0.847,over,under,1,10.48,0.524,Med,Med
311,Evan Mobley,Alperen Sengun,19.5,23.5,14.84,28.14,0.776,0.762,under,over,1,7.38,0.369,High,High
543,Bennedict Mathurin,Dillon Brooks,20.5,18.5,25.16,22.80,0.768,0.743,over,over,1,6.77,0.339,High,High
2656,Isaac Okoro,Lauri Markkanen,7.5,24.5,11.22,28.82,0.733,0.734,over,over,0,5.82,0.291,Med,High
3374,Julius Randle,Buddy Hield,22.5,7.5,26.17,10.83,0.707,0.724,over,over,0,5.05,0.253,High,Med
43,Pascal Siakam,Saddiq Bey,22.5,8.5,25.36,11.87,0.665,0.700,over,over,0,3.70,0.185,High,High
370,Andrew Nembhard,D'Angelo Russell,15.5,12.5,18.42,15.66,0.663,0.684,over,over,0,3.32,0.166,High,High
1898,Khris Middleton,Cooper Flagg,10.5,17.5,12.73,14.46,0.663,0.683,over,under,0,3.30,0.165,Med,High
3247,Jeremiah Fears,Keyonte George,14.5,18.5,17.24,21.23,0.656,0.654,over,over,0,2.63,0.131,High,High
3153,Naji Marshall,Nikola Jokić,11.5,28.5,13.91,30.83,0.655,0.653,over,over,0,2.57,0.129,High,Med


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 123 players...
Processing 111 players...
Generated 5803 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV$,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
4054,Tre Jones,Jerami Grant,9.5,22.5,14.89,17.10,over,under,0.823,0.847,0.6827,0.245,0.269,0.362,10.48,0.524,1,5.82,5.28,Med,Med,"(3.5, 26.3)","(6.8, 27.5)",0.05,0,104.8
281,Evan Mobley,Alperen Sengun,19.5,23.5,14.84,28.14,under,over,0.776,0.762,0.5793,0.198,0.184,0.257,7.38,0.369,1,6.15,6.51,High,High,"(2.8, 26.9)","(15.4, 40.9)",0.05,0,73.8
161,Bennedict Mathurin,Julius Randle,20.5,21.5,25.16,26.17,over,over,0.768,0.756,0.5691,0.190,0.178,0.247,7.07,0.354,1,6.36,6.74,High,High,"(12.7, 37.6)","(13.0, 39.4)",0.05,0,70.7
4988,Dillon Brooks,Lauri Markkanen,18.5,24.5,22.80,28.82,over,over,0.743,0.734,0.5342,0.165,0.156,0.211,6.03,0.301,1,6.60,6.92,High,High,"(9.9, 35.7)","(15.3, 42.4)",0.05,0,60.3
4108,Isaac Okoro,Aaron Gordon,7.5,16.5,11.22,19.75,over,over,0.733,0.732,0.5262,0.155,0.154,0.203,5.79,0.289,0,5.97,5.24,Med,Med,"(0.0, 22.9)","(9.5, 30.0)",0.05,0,57.9
5523,Cameron Johnson,Buddy Hield,12.5,7.5,9.26,10.83,under,over,0.724,0.724,0.5141,0.146,0.146,0.190,5.42,0.271,0,5.44,5.60,Med,Med,"(0.0, 19.9)","(0.0, 21.8)",0.05,0,54.2
52,Pascal Siakam,Saddiq Bey,22.5,8.5,25.36,11.87,over,over,0.665,0.700,0.4567,0.087,0.122,0.132,3.70,0.185,0,6.69,6.41,High,High,"(12.3, 38.5)","(0.0, 24.4)",0.05,0,37.0
2794,Khris Middleton,D'Angelo Russell,10.5,12.5,12.73,15.66,over,over,0.663,0.684,0.4441,0.085,0.106,0.119,3.32,0.166,0,5.31,6.60,Med,High,"(2.3, 23.1)","(2.7, 28.6)",0.05,0,33.2
460,Andrew Nembhard,Cooper Flagg,15.5,17.5,18.42,14.46,over,under,0.663,0.683,0.4434,0.085,0.105,0.118,3.30,0.165,0,6.95,6.39,High,High,"(4.8, 32.0)","(1.9, 27.0)",0.05,0,33.0
4441,Jeremiah Fears,Keyonte George,14.5,18.5,17.24,21.23,over,over,0.656,0.654,0.4209,0.078,0.076,0.095,2.63,0.131,0,6.80,6.88,High,High,"(3.9, 30.6)","(7.8, 34.7)",0.05,0,26.3


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 99 players...
Processing 91 players...
Generated 120197 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
13851,Evan Mobley,Tre Jones,Jerami Grant,19.5,9.5,22.5,14.84,14.89,17.10,0.776,0.823,0.847,under,over,under,1,19.19,0.384,High,Med,Med
25320,Bennedict Mathurin,Dillon Brooks,Alperen Sengun,20.5,18.5,23.5,25.16,22.80,28.14,0.768,0.743,0.762,over,over,over,1,13.48,0.270,High,High,High
98566,Isaac Okoro,Buddy Hield,Lauri Markkanen,7.5,7.5,24.5,11.22,10.83,28.82,0.733,0.724,0.734,over,over,over,0,11.05,0.221,Med,Med,High
104528,Cooper Flagg,D'Angelo Russell,Julius Randle,17.5,12.5,22.5,14.46,15.66,26.17,0.683,0.684,0.707,under,over,over,0,7.83,0.157,High,High,High
268,Pascal Siakam,Andrew Nembhard,Saddiq Bey,22.5,15.5,8.5,25.36,18.42,11.87,0.665,0.663,0.700,over,over,over,0,6.68,0.134,High,High,High


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 123 players...
Processing 111 players...
Generated 219559 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
15638,Evan Mobley,Tre Jones,Jerami Grant,19.5,9.5,22.5,14.84,14.89,17.10,0.776,0.823,0.847,under,over,under,1,19.19,0.384,High,Med,Med
10823,Bennedict Mathurin,Julius Randle,Alperen Sengun,20.5,21.5,23.5,25.16,26.17,28.14,0.768,0.756,0.762,over,over,over,1,13.90,0.278,High,High,High
183888,Isaac Okoro,Dillon Brooks,Lauri Markkanen,7.5,18.5,24.5,11.22,22.80,28.82,0.733,0.743,0.734,over,over,over,0,11.58,0.232,Med,High,High
216687,Aaron Gordon,Cameron Johnson,Buddy Hield,16.5,12.5,7.5,19.75,9.26,10.83,0.732,0.724,0.724,over,under,over,0,10.74,0.215,Med,Med,Med
4243,Pascal Siakam,Cooper Flagg,D'Angelo Russell,22.5,17.5,12.5,25.36,14.46,15.66,0.665,0.683,0.684,over,under,over,0,6.78,0.136,High,High,High


In [10]:
# playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)